<a href="https://colab.research.google.com/github/AmanUllah2011/churn-prediction-ml/blob/main/Hour_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print(" আমি ML শিখছি হালাল ইনকামের জন্য")

 আমি ML শিখছি হালাল ইনকামের জন্য


In [2]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv')
print(df.shape)
print(df.head())

(7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Co

In [3]:
# কতজন Customerচলে গেছে, কতজন আছে
print(df['Churn'].value_counts())
#মোট কত % চলে গেছে
print(df['Churn'].value_counts(normalize = True)*100)

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


In [4]:
#কোন কলামে মিসিং ডাটা আছে
print(df.isnull().sum())
#data এর ধরন দেখো
print(df.dtypes)

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      f

In [5]:
# TotalCharges-কে number-এ convert করো
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# এখন কতটা missing হলো দেখো
print(df['TotalCharges'].isnull().sum())

11


In [6]:
# 11টা missing row বাদ দাও
df = df.dropna()

# confirm করো
print(df.shape)

(7032, 21)


In [7]:
# Churn: Yes=1, No=0
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# customerID দরকার নেই, বাদ দাও
df = df.drop('customerID', axis=1)

# বাকি text column গুলো number-এ convert করো
df = pd.get_dummies(df, drop_first=True)

print(df.shape)
print(df.head(2))

(7032, 31)
   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  Churn  gender_Male  \
0              0       1           29.85         29.85      0        False   
1              0      34           56.95       1889.50      0         True   

   Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0         True           False             False   
1        False           False              True   

   MultipleLines_No phone service  ...  StreamingTV_No internet service  \
0                            True  ...                            False   
1                           False  ...                            False   

   StreamingTV_Yes  StreamingMovies_No internet service  StreamingMovies_Yes  \
0            False                                False                False   
1            False                                False                False   

   Contract_One year  Contract_Two year  PaperlessBilling_Yes  \
0              False              False                  True   


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# X = features, y = target
X = df.drop('Churn', axis=1)
y = df['Churn']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model train করো
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Accuracy দেখো
predictions = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 0.7853589196872779


In [9]:
import pandas as pd

# কোন factor সবচেয়ে important
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance.head(10))

TotalCharges                      0.193409
MonthlyCharges                    0.169758
tenure                            0.167572
InternetService_Fiber optic       0.039999
PaymentMethod_Electronic check    0.035016
OnlineSecurity_Yes                0.028905
Contract_Two year                 0.028618
gender_Male                       0.026971
TechSupport_Yes                   0.025829
PaperlessBilling_Yes              0.025044
dtype: float64


In [10]:
import joblib

joblib.dump(model, 'churn_model.pkl')
print("Model saved!")

Model saved!


In [11]:
from google.colab import files
files.download('churn_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
from sklearn.metrics import confusion_matrix, classification_report

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.63      0.48      0.54       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.70      1407
weighted avg       0.77      0.79      0.78      1407



In [13]:
from sklearn.ensemble import GradientBoostingClassifier

# উন্নত model
model2 = GradientBoostingClassifier(random_state=42)
model2.fit(X_train, y_train)

predictions2 = model2.predict(X_test)
print(classification_report(y_test, predictions2))

              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.64      0.48      0.55       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407



In [14]:
from sklearn.ensemble import RandomForestClassifier

# class_weight দিয়ে চলে যাওয়া customer-দের বেশি গুরুত্ব দাও
model3 = RandomForestClassifier(
    class_weight='balanced',
    random_state=42
)
model3.fit(X_train, y_train)

predictions3 = model3.predict(X_test)
print(classification_report(y_test, predictions3))

              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1033
           1       0.63      0.45      0.53       374

    accuracy                           0.78      1407
   macro avg       0.73      0.68      0.69      1407
weighted avg       0.77      0.78      0.77      1407



In [15]:
import joblib

# প্রথম model-ই রাখি
joblib.dump(model, 'churn_model_final.pkl')

# Client-এর জন্য prediction function
def predict_churn(data_dict):
    import pandas as pd
    df_new = pd.DataFrame([data_dict])
    df_new = pd.get_dummies(df_new)
    df_new = df_new.reindex(columns=X.columns, fill_value=0)
    result = model.predict(df_new)
    return "চলে যাবে ❌" if result[0] == 1 else "থাকবে ✅"

print("Function ready!")

Function ready!


In [16]:
# একজন নতুন customer-এর data দিয়ে test করো
test_customer = {
    'SeniorCitizen': 0,
    'tenure': 2,
    'MonthlyCharges': 70.0,
    'TotalCharges': 140.0
}

print(predict_churn(test_customer))

থাকবে ✅


In [17]:
# Notebook-এর নাম দাও
# File > Download > Download .ipynb
print("Colab থেকে notebook download করো")
print("File menu > Download > Download .ipynb")

Colab থেকে notebook download করো
File menu > Download > Download .ipynb
